# MagicFinance — Module E: Position Sizing & Portfolio Engine

**Goal**: Combine Module A signals + Module D forecasts into optimised portfolios  
using Markowitz mean-variance optimisation.

**A/B Comparison**:
- **Path A (Fixed Weights)**: `0.4 × thesis_score + 0.4 × forecast_probability + 0.2 × confidence`
- **Path B (Dynamic Weights)**: Qwen 9B reasons about optimal weights per ticker

**Pipeline**:
1. Load signals from Module A (Qdrant) + forecasts from Module D (Qdrant)
2. Build composite expected returns — both fixed and dynamic methods
3. Build stock universe (Nanobot watchlist + top Module A tickers)
4. Fetch historical prices via yfinance → covariance matrix
5. Markowitz optimisation for both methods
6. A/B comparison: portfolios, metrics, allocation charts
7. Backtest: portfolio P&L vs S&P 500 benchmark
8. Fire Slack alert with final portfolio

**Requires**: Modules A and D must have run first (signals in Qdrant)

## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(name)s: %(message)s')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from datetime import datetime, timedelta
from dotenv import load_dotenv
from IPython.display import display

load_dotenv(dotenv_path='../.env')

from magicfinance import config
from magicfinance import qdrant_client as qc
from magicfinance.portfolio import (
    compute_expected_return_fixed,
    compute_expected_return_dynamic,
    scale_expected_returns,
    optimize_portfolio,
    portfolio_metrics,
    build_portfolio_positions,
)
from magicfinance.yfinance_client import (
    fetch_prices,
    compute_covariance_matrix,
    backtest_portfolio,
    benchmark_sp500,
    get_ticker_info,
)
from magicfinance.slack_client import alert_portfolio_ready

# ─── Nanobot watchlist (base universe from stock_watchdog.py) ────────────────
# Update this list to match your current Nanobot watchlist
NANOBOT_WATCHLIST = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA', 'JPM', 'V', 'UNH']

print(f'Nanobot watchlist: {NANOBOT_WATCHLIST}')
print(f'Portfolio value: ${config.PORTFOLIO_VALUE_USD:,.0f}')

## 1. Load Signals from Qdrant

In [ ]:
# Load investable Module A signals
try:
    a_signals = qc.get_investable_signals(min_confidence=0.60)
    print(f'Module A signals loaded: {len(a_signals)}')
except Exception as e:
    print(f'Qdrant error: {e}')
    a_signals = []

# Load Module D forecasts (per-stock only)
try:
    d_forecasts_all = qc.get_forecast_history()
    d_forecasts = [f for f in d_forecasts_all if not f.get('is_macro_event', False) and f.get('ticker')]
    print(f'Module D stock forecasts loaded: {len(d_forecasts)}')
except Exception as e:
    print(f'Qdrant error: {e}')
    d_forecasts = []

# Build forecast lookup: ticker → highest probability forecast
forecast_by_ticker = {}
for f in d_forecasts:
    ticker = f.get('ticker')
    if ticker:
        existing = forecast_by_ticker.get(ticker, {})
        if f.get('forecast_probability', 0) > existing.get('forecast_probability', 0):
            forecast_by_ticker[ticker] = f

print(f'Tickers with forecasts: {list(forecast_by_ticker.keys())}')

## 2. Build Stock Universe

In [ ]:
# Top investable tickers from Module A
top_reddit_tickers = sorted(a_signals, key=lambda s: s.get('confidence_level', 0), reverse=True)
top_reddit_tickers = [s['ticker'] for s in top_reddit_tickers[:10]]  # top 10

# Combined universe: Nanobot watchlist + top Reddit tickers
universe = list(dict.fromkeys(NANOBOT_WATCHLIST + top_reddit_tickers))  # preserve order, deduplicate
print(f'Stock universe ({len(universe)} tickers): {universe}')

## 3. Compute Expected Returns

In [ ]:
# Build signal map for universe tickers
signal_by_ticker = {s['ticker']: s for s in a_signals}

# Default values for tickers in watchlist but not in Reddit signals
DEFAULT_THESIS = 0.5
DEFAULT_CONFIDENCE = 0.5
DEFAULT_FORECAST_PROB = 0.5  # neutral forecast

# ─── Path A: Fixed Weights ───────────────────────────────────────────────────
signals_fixed = []
for ticker in universe:
    sig = signal_by_ticker.get(ticker, {})
    fc = forecast_by_ticker.get(ticker, {})
    
    thesis = sig.get('thesis_score', DEFAULT_THESIS)
    confidence = sig.get('confidence_level', DEFAULT_CONFIDENCE)
    forecast_prob = fc.get('forecast_probability', DEFAULT_FORECAST_PROB)
    
    er = compute_expected_return_fixed(thesis, forecast_prob, confidence)
    signals_fixed.append({
        'ticker': ticker,
        'thesis_score': thesis,
        'confidence_level': confidence,
        'forecast_probability': forecast_prob,
        'expected_return': er,
        'from_reddit': ticker in signal_by_ticker,
        'has_forecast': ticker in forecast_by_ticker,
    })

df_fixed = pd.DataFrame(signals_fixed)
raw_er_fixed = pd.Series(df_fixed['expected_return'].values, index=df_fixed['ticker'])
er_fixed = scale_expected_returns(raw_er_fixed)

print('Path A (Fixed Weights) — Expected Returns:')
display(df_fixed.set_index('ticker')[['thesis_score','forecast_probability','confidence_level','expected_return']].style.format('{:.0%}').background_gradient(subset=['expected_return'], cmap='RdYlGn'))

In [ ]:
# ─── Path B: Dynamic Weights (Qwen 9B) ──────────────────────────────────────
print(f'Computing dynamic weights with {config.MODEL_9B}...')
print('(This may take 2-5 minutes for the full universe)')

signals_dynamic = []
weight_details = []

for ticker in universe:
    sig = signal_by_ticker.get(ticker, {})
    fc = forecast_by_ticker.get(ticker, {})
    
    thesis = sig.get('thesis_score', DEFAULT_THESIS)
    confidence = sig.get('confidence_level', DEFAULT_CONFIDENCE)
    forecast_prob = fc.get('forecast_probability', DEFAULT_FORECAST_PROB)
    
    try:
        er, weights_used = compute_expected_return_dynamic(ticker, thesis, forecast_prob, confidence)
        weight_details.append({'ticker': ticker, **weights_used})
        print(f'  {ticker}: er={er:.2f} | weights={weights_used.get("weight_thesis",0):.2f}/{weights_used.get("weight_forecast",0):.2f}/{weights_used.get("weight_confidence",0):.2f}')
    except Exception as e:
        # Fall back to fixed weights if LLM fails
        er = compute_expected_return_fixed(thesis, forecast_prob, confidence)
        weight_details.append({'ticker': ticker, 'weight_thesis': config.FIXED_WEIGHTS['thesis'], 
                                'weight_forecast': config.FIXED_WEIGHTS['forecast'],
                                'weight_confidence': config.FIXED_WEIGHTS['confidence'],
                                'reasoning': f'fallback (LLM error: {e})'})
        print(f'  {ticker}: LLM failed, using fixed weights')
    
    signals_dynamic.append({
        'ticker': ticker,
        'thesis_score': thesis,
        'confidence_level': confidence,
        'forecast_probability': forecast_prob,
        'expected_return': er,
    })

df_dynamic = pd.DataFrame(signals_dynamic)
raw_er_dynamic = pd.Series(df_dynamic['expected_return'].values, index=df_dynamic['ticker'])
er_dynamic = scale_expected_returns(raw_er_dynamic)

print('\nPath B (Dynamic Weights) — Expected Returns:')
display(df_dynamic.set_index('ticker')[['expected_return']].rename(columns={'expected_return': 'dynamic_er'}).join(
    df_fixed.set_index('ticker')[['expected_return']].rename(columns={'expected_return': 'fixed_er'})
).style.format('{:.0%}').background_gradient(cmap='RdYlGn'))

## 4. Fetch Historical Prices + Covariance Matrix

In [ ]:
print(f'Fetching {config.LOOKBACK_DAYS} days of price data for {len(universe)} tickers...')
prices = fetch_prices(universe, lookback_days=config.LOOKBACK_DAYS)

print(f'Price data: {prices.shape[0]} trading days × {prices.shape[1]} tickers')
print(f'Date range: {prices.index[0].date()} → {prices.index[-1].date()}')

cov_matrix = compute_covariance_matrix(prices)
print(f'Covariance matrix: {cov_matrix.shape}')

# Only keep tickers with price data
valid_tickers = [t for t in universe if t in prices.columns]
er_fixed = er_fixed[er_fixed.index.isin(valid_tickers)]
er_dynamic = er_dynamic[er_dynamic.index.isin(valid_tickers)]
print(f'Tickers with price data: {len(valid_tickers)}')

## 5. Markowitz Optimisation — Both Methods

In [ ]:
# Path A: Fixed weights → optimisation
weights_fixed = optimize_portfolio(
    expected_returns=er_fixed,
    cov_matrix=cov_matrix,
    risk_free_rate=config.RISK_FREE_RATE,
    max_position=config.MAX_POSITION_PCT,
    min_position=config.MIN_POSITION_PCT,
)
metrics_fixed = portfolio_metrics(weights_fixed, er_fixed, cov_matrix)

# Path B: Dynamic weights → optimisation
weights_dynamic = optimize_portfolio(
    expected_returns=er_dynamic,
    cov_matrix=cov_matrix,
    risk_free_rate=config.RISK_FREE_RATE,
    max_position=config.MAX_POSITION_PCT,
    min_position=config.MIN_POSITION_PCT,
)
metrics_dynamic = portfolio_metrics(weights_dynamic, er_dynamic, cov_matrix)

# Build position tables
positions_fixed = build_portfolio_positions(weights_fixed, signals_fixed, config.PORTFOLIO_VALUE_USD)
positions_dynamic = build_portfolio_positions(weights_dynamic, signals_dynamic, config.PORTFOLIO_VALUE_USD)

# Display comparison
comparison = pd.DataFrame({
    'Fixed Weights': metrics_fixed,
    'Dynamic Weights (Qwen)': metrics_dynamic,
}).T
comparison.columns = ['Expected Return', 'Volatility', 'Sharpe Ratio']

print('\n📊 Portfolio Metrics Comparison:')
display(comparison.style.format({
    'Expected Return': '{:.1%}',
    'Volatility': '{:.1%}',
    'Sharpe Ratio': '{:.3f}',
}).background_gradient(subset=['Sharpe Ratio'], cmap='RdYlGn'))

## 6. A/B Portfolio Comparison — Allocation Charts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('MagicFinance — Module E: Portfolio Allocation (A/B Comparison)', fontsize=14, fontweight='bold')

colors = plt.cm.Set3.colors

for ax, positions, title in [
    (axes[0], positions_fixed, f'Path A: Fixed Weights\nER={metrics_fixed["expected_return"]:.1%} | Vol={metrics_fixed["volatility"]:.1%} | Sharpe={metrics_fixed["sharpe_ratio"]:.2f}'),
    (axes[1], positions_dynamic, f'Path B: Dynamic Weights (Qwen 9B)\nER={metrics_dynamic["expected_return"]:.1%} | Vol={metrics_dynamic["volatility"]:.1%} | Sharpe={metrics_dynamic["sharpe_ratio"]:.2f}'),
]:
    if positions:
        tickers = [p['ticker'] for p in positions]
        allocs = [p['allocation_pct'] for p in positions]
        ax.pie(allocs, labels=tickers, autopct='%1.1f%%', colors=colors[:len(tickers)], startangle=90)
    ax.set_title(title)

plt.tight_layout()
plt.savefig('../data/module_e_portfolios.png', dpi=150, bbox_inches='tight')
plt.show()

# Side-by-side position table
df_pos_fixed = pd.DataFrame(positions_fixed).set_index('ticker')
df_pos_dynamic = pd.DataFrame(positions_dynamic).set_index('ticker')

df_compare = df_pos_fixed[['allocation_pct']].rename(columns={'allocation_pct': 'fixed_%'}).join(
    df_pos_dynamic[['allocation_pct']].rename(columns={'allocation_pct': 'dynamic_%'}),
    how='outer'
).fillna(0)
df_compare['delta'] = df_compare['dynamic_%'] - df_compare['fixed_%']

print('\nAllocation Differences (Dynamic vs Fixed):')
display(df_compare.style.format('{:.1%}').background_gradient(subset=['delta'], cmap='RdYlGn'))

## 7. Backtest: Portfolio P&L vs S&P 500

In [ ]:
# Backtest uses current signals applied to historical price window
# This answers: "if we had held these positions for the past year, what would P&L look like?"
backtest_start = (datetime.today() - timedelta(days=252)).strftime('%Y-%m-%d')

bt_fixed = backtest_portfolio(weights_fixed, prices, config.PORTFOLIO_VALUE_USD)
bt_dynamic = backtest_portfolio(weights_dynamic, prices, config.PORTFOLIO_VALUE_USD)
bt_spy = benchmark_sp500(start_date=backtest_start, initial_value=config.PORTFOLIO_VALUE_USD)

# Align all to same date range
bt_start = max(bt_fixed.index.min(), bt_dynamic.index.min(), bt_spy.index.min() if not bt_spy.empty else bt_fixed.index.min())
bt_fixed = bt_fixed[bt_fixed.index >= bt_start]
bt_dynamic = bt_dynamic[bt_dynamic.index >= bt_start]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('MagicFinance — Backtest: Portfolio P&L vs S&P 500', fontsize=14, fontweight='bold')

# Portfolio value over time
ax1 = axes[0]
ax1.plot(bt_fixed.index, bt_fixed['portfolio_value'], label='Fixed Weights', color='steelblue', linewidth=2)
ax1.plot(bt_dynamic.index, bt_dynamic['portfolio_value'], label='Dynamic Weights (Qwen)', color='darkorange', linewidth=2)
if not bt_spy.empty:
    ax1.plot(bt_spy.index, bt_spy['portfolio_value'], label='S&P 500', color='gray', linewidth=1.5, linestyle='--')
ax1.set_title(f'Portfolio Value (starting ${config.PORTFOLIO_VALUE_USD:,.0f})')
ax1.set_ylabel('USD')
ax1.legend()
ax1.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Cumulative returns
ax2 = axes[1]
ax2.plot(bt_fixed.index, bt_fixed['cumulative_return'], label='Fixed Weights', color='steelblue', linewidth=2)
ax2.plot(bt_dynamic.index, bt_dynamic['cumulative_return'], label='Dynamic Weights (Qwen)', color='darkorange', linewidth=2)
if not bt_spy.empty:
    ax2.plot(bt_spy.index, bt_spy['cumulative_return'], label='S&P 500', color='gray', linewidth=1.5, linestyle='--')
ax2.axhline(0, color='black', linewidth=0.5)
ax2.set_title('Cumulative Return')
ax2.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax2.legend()

plt.tight_layout()
plt.savefig('../data/module_e_backtest.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary metrics
def bt_summary(bt_df, name):
    if bt_df.empty:
        return {}
    total_return = bt_df['cumulative_return'].iloc[-1]
    max_drawdown = (bt_df['portfolio_value'] / bt_df['portfolio_value'].cummax() - 1).min()
    volatility = bt_df['daily_return'].std() * (252 ** 0.5)
    return {'Strategy': name, 'Total Return': total_return, 'Max Drawdown': max_drawdown, 'Annual Vol': volatility}

summaries = [bt_summary(bt_fixed, 'Fixed Weights'), bt_summary(bt_dynamic, 'Dynamic Weights (Qwen)')]
if not bt_spy.empty:
    summaries.append(bt_summary(bt_spy, 'S&P 500'))
    
display(pd.DataFrame(summaries).set_index('Strategy').style.format({
    'Total Return': '{:.1%}', 'Max Drawdown': '{:.1%}', 'Annual Vol': '{:.1%}'
}))

## 8. Final Portfolio — Fire Slack Alert

In [ ]:
# Choose method with better Sharpe to send as primary alert
if metrics_fixed['sharpe_ratio'] >= metrics_dynamic['sharpe_ratio']:
    best_portfolio = positions_fixed
    best_method = 'fixed_weights'
    print(f'Best portfolio: Fixed Weights (Sharpe={metrics_fixed["sharpe_ratio"]:.3f})')
else:
    best_portfolio = positions_dynamic
    best_method = 'dynamic_weights'
    print(f'Best portfolio: Dynamic Weights (Sharpe={metrics_dynamic["sharpe_ratio"]:.3f})')

# Fire Slack alert
sent = alert_portfolio_ready(best_portfolio, best_method, config.PORTFOLIO_VALUE_USD)
print(f'Slack alert: {"✅ sent" if sent else "❌ not sent (check SLACK_WEBHOOK_URL)"}')

# Final position table
print('\n💼 Final Portfolio:')
display(pd.DataFrame(best_portfolio).style.format({
    'allocation_pct': '{:.1%}',
    'usd_value': '${:,.0f}',
    'expected_return': '{:.1%}',
    'confidence_level': '{:.0%}',
    'thesis_score': '{:.0%}',
    'forecast_probability': '{:.0%}',
}).background_gradient(subset=['allocation_pct'], cmap='Blues'))

print('\n✅ MagicFinance pipeline complete.')
print(f'Signals: {len(a_signals)} | Forecasts: {len(d_forecasts)} | Positions: {len(best_portfolio)}')